# Kaggle FF++ / Celeb-DF external visual validation

"
       "This evaluates existing SDFVD 2.0 visual checkpoints on the Kaggle dataset's **test split only**. It does not train or tune on Kaggle. The Kaggle archive stays in Drive; extracted files and temporary source files stay in Colab runtime storage. Results are image-level because source-video mapping is not established.

"
       "Before running: set `BASE` and `CV_RUN_FOLDER` to your Drive locations, then add Kaggle API credentials as Colab Secrets named `KAGGLE_USERNAME` and `KAGGLE_KEY`. These are needed to download the dataset; never paste credentials into notebook cells or save them in Drive.


In [ ]:
from google.colab import drive, userdata
from pathlib import Path
import json, os, shutil, subprocess, sys, time, zipfile

drive.mount('/content/drive')
# Change this one path in Colab to the Drive folder containing the dataset artifacts.
BASE = Path('/content/drive/MyDrive/Datasets')
DEST = BASE / 'kaggle_mahsan8381_external_validation'
DEST.mkdir(parents=True, exist_ok=True)
CV_RUN_FOLDER = 'REPLACE_WITH_YOUR_SDFVD2_CV_RUN_FOLDER'
CV_ROOT = BASE / CV_RUN_FOLDER / 'group_cv'
SOURCE_B64 = BASE / 'poor_guys_deepfake_colab_source.b64'
assert CV_ROOT.is_dir(), f'Missing saved SDFVD2 fold checkpoints: {CV_ROOT}'
assert SOURCE_B64.is_file(), f'Missing project source bundle: {SOURCE_B64}'

username = userdata.get('KAGGLE_USERNAME')
api_key = userdata.get('KAGGLE_KEY')
if not username or not api_key:
    raise RuntimeError('Add Kaggle API credentials in Colab Secrets as KAGGLE_USERNAME and KAGGLE_KEY, then rerun this cell.')
kaggle_cfg = Path.home() / '.kaggle'
kaggle_cfg.mkdir(parents=True, exist_ok=True)
(kaggle_cfg / 'kaggle.json').write_text(json.dumps({'username': username, 'key': api_key}), encoding='utf-8')
os.chmod(kaggle_cfg / 'kaggle.json', 0o600)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'kaggle'], check=True)

archives = list(DEST.glob('deepfake-dataset*.zip'))
if not archives:
    subprocess.run(['kaggle', 'datasets', 'download', '-d', 'mahsan8381/deepfake-dataset', '-p', str(DEST)], check=True)
    archives = list(DEST.glob('deepfake-dataset*.zip'))
if len(archives) != 1:
    raise RuntimeError(f'Expected one Kaggle archive in {DEST}, found: {[p.name for p in archives]}')
ARCHIVE = archives[0]
print('Kaggle archive stored in Drive:', ARCHIVE, '| bytes:', ARCHIVE.stat().st_size)
print('Existing checkpoints:', CV_ROOT)


In [ ]:
# Extract safely to ephemeral Colab storage. The ZIP remains in Drive for reproducibility.
WORK = Path('/content/kaggle_mahsan8381_external_validation')
EXTRACT_ROOT = WORK / 'extracted'
DATASET_ROOT = WORK / 'dataset'
if not (WORK / 'extraction_complete').is_file():
    WORK.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(ARCHIVE) as zf:
        base = EXTRACT_ROOT.resolve()
        required = sum(info.file_size for info in zf.infolist())
        free = shutil.disk_usage('/content').free
        if free < required + 1024**3:
            raise RuntimeError(f'Not enough Colab temporary disk to extract safely: need about {required + 1024**3} bytes, have {free}')
        for item in zf.infolist():
            target = (EXTRACT_ROOT / item.filename).resolve()
            if target != base and base not in target.parents:
                raise RuntimeError(f'Unsafe path in Kaggle archive: {item.filename}')
        zf.extractall(EXTRACT_ROOT)
    tests = [p for p in EXTRACT_ROOT.rglob('test') if p.is_dir() and any((p / label).is_dir() for label in ('real', 'fake'))]
    if len(tests) != 1:
        raise RuntimeError(f'Could not identify a unique dataset/test directory with real/fake folders: {tests[:10]}')
    TEST_ROOT = tests[0]
    DATASET_ROOT = TEST_ROOT.parent
    (WORK / 'dataset_root.txt').write_text(str(DATASET_ROOT), encoding='utf-8')
    (WORK / 'test_root.txt').write_text(str(TEST_ROOT), encoding='utf-8')
    (WORK / 'extraction_complete').write_text('ok', encoding='utf-8')
else:
    DATASET_ROOT = Path((WORK / 'dataset_root.txt').read_text(encoding='utf-8'))
    TEST_ROOT = Path((WORK / 'test_root.txt').read_text(encoding='utf-8'))
print('Dataset root:', DATASET_ROOT)
print('Test-only evaluation root:', TEST_ROOT)


In [ ]:
# Recover the project's model implementation from the Drive source bundle.
import base64
EVALUATOR_B64 = 'IiIiRXZhbHVhdGUgU0RGVkQyIHZpc3VhbCBDViBjaGVja3BvaW50cyBvbiBLYWdnbGUgRkYrKy9DZWxlYi1ERiB0ZXN0IGZyYW1lcy4KClRoaXMgaXMgYSB0ZXN0LW9ubHkgZXh0ZXJuYWwgZXZhbHVhdGlvbi4gSXQgbmV2ZXIgdHVuZXMgb24gdGhlIEthZ2dsZSBkYXRhOwplYWNoIGZvbGQgdXNlcyB0aGUgdGhyZXNob2xkIHNlbGVjdGVkIG9uIHRoYXQgZm9sZCdzIFNERlZEMiB2YWxpZGF0aW9uIHNldC4KVGhlIHNjb3JlIHVuaXQgaXMgYW4gaW1hZ2UsIG5vdCBhbiBpbmRlcGVuZGVudCBzb3VyY2UgdmlkZW8uCiIiIgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgppbXBvcnQgYXJncGFyc2UKaW1wb3J0IGNzdgppbXBvcnQgaGFzaGxpYgppbXBvcnQganNvbgppbXBvcnQgbWF0aAppbXBvcnQgc3RhdGlzdGljcwppbXBvcnQgc3lzCmZyb20gY29sbGVjdGlvbnMgaW1wb3J0IENvdW50ZXIsIGRlZmF1bHRkaWN0CmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aAoKaW1wb3J0IHRvcmNoCmZyb20gUElMIGltcG9ydCBJbWFnZQpmcm9tIHRvcmNoLnV0aWxzLmRhdGEgaW1wb3J0IERhdGFMb2FkZXIsIERhdGFzZXQKZnJvbSB0b3JjaHZpc2lvbiBpbXBvcnQgdHJhbnNmb3JtcwoKZnJvbSBtb2RlbCBpbXBvcnQgYnVpbGRfbW9kZWwKCklNQUdFX0VYVEVOU0lPTlMgPSB7Ii5qcGciLCAiLmpwZWciLCAiLnBuZyIsICIud2VicCIsICIuYm1wIn0KTEFCRUxfRElSUyA9IHsicmVhbCI6IDAsICJmYWtlIjogMSwgImRlZXBmYWtlIjogMSwgInN5bnRoZXRpYyI6IDF9CgoKZGVmIGF1YyhzY29yZXM6IGxpc3RbZmxvYXRdLCBsYWJlbHM6IGxpc3RbaW50XSkgLT4gZmxvYXQgfCBOb25lOgogICAgcG9zaXRpdmVzID0gc3VtKGxhYmVscykKICAgIG5lZ2F0aXZlcyA9IGxlbihsYWJlbHMpIC0gcG9zaXRpdmVzCiAgICBpZiBwb3NpdGl2ZXMgPT0gMCBvciBuZWdhdGl2ZXMgPT0gMDoKICAgICAgICByZXR1cm4gTm9uZQogICAgb3JkZXJlZCA9IHNvcnRlZCh6aXAoc2NvcmVzLCBsYWJlbHMpKQogICAgcmFua19zdW0gPSAwLjAKICAgIGkgPSAwCiAgICB3aGlsZSBpIDwgbGVuKG9yZGVyZWQpOgogICAgICAgIGogPSBpICsgMQogICAgICAgIHdoaWxlIGogPCBsZW4ob3JkZXJlZCkgYW5kIG9yZGVyZWRbal1bMF0gPT0gb3JkZXJlZFtpXVswXToKICAgICAgICAgICAgaiArPSAxCiAgICAgICAgYXZlcmFnZV9yYW5rID0gKGkgKyAxICsgaikgLyAyCiAgICAgICAgcmFua19zdW0gKz0gc3VtKGxhYmVsIGZvciBfLCBsYWJlbCBpbiBvcmRlcmVkW2k6al0pICogYXZlcmFnZV9yYW5rCiAgICAgICAgaSA9IGoKICAgIHJldHVybiAocmFua19zdW0gLSBwb3NpdGl2ZXMgKiAocG9zaXRpdmVzICsgMSkgLyAyKSAvIChwb3NpdGl2ZXMgKiBuZWdhdGl2ZXMpCgoKZGVmIHNjb3JlX21ldHJpY3Moc2NvcmVzOiBsaXN0W2Zsb2F0XSwgbGFiZWxzOiBsaXN0W2ludF0sIHRocmVzaG9sZDogZmxvYXQpIC0+IGRpY3Q6CiAgICBwcmVkaWN0ZWQgPSBbaW50KHNjb3JlID49IHRocmVzaG9sZCkgZm9yIHNjb3JlIGluIHNjb3Jlc10KICAgIHRwID0gc3VtKHAgPT0gMSBhbmQgeSA9PSAxIGZvciBwLCB5IGluIHppcChwcmVkaWN0ZWQsIGxhYmVscykpCiAgICB0biA9IHN1bShwID09IDAgYW5kIHkgPT0gMCBmb3IgcCwgeSBpbiB6aXAocHJlZGljdGVkLCBsYWJlbHMpKQogICAgZnAgPSBzdW0ocCA9PSAxIGFuZCB5ID09IDAgZm9yIHAsIHkgaW4gemlwKHByZWRpY3RlZCwgbGFiZWxzKSkKICAgIGZuID0gc3VtKHAgPT0gMCBhbmQgeSA9PSAxIGZvciBwLCB5IGluIHppcChwcmVkaWN0ZWQsIGxhYmVscykpCiAgICBwb3NpdGl2ZXMsIG5lZ2F0aXZlcyA9IHN1bShsYWJlbHMpLCBsZW4obGFiZWxzKSAtIHN1bShsYWJlbHMpCiAgICB0cHIgPSB0cCAvIHBvc2l0aXZlcyBpZiBwb3NpdGl2ZXMgZWxzZSBOb25lCiAgICB0bnIgPSB0biAvIG5lZ2F0aXZlcyBpZiBuZWdhdGl2ZXMgZWxzZSBOb25lCiAgICByZXR1cm4gewogICAgICAgICJjb3VudCI6IGxlbihsYWJlbHMpLAogICAgICAgICJjbGFzc19jb3VudHMiOiB7InJlYWwiOiBuZWdhdGl2ZXMsICJzeW50aGV0aWMiOiBwb3NpdGl2ZXN9LAogICAgICAgICJhY2N1cmFjeSI6ICh0cCArIHRuKSAvIGxlbihsYWJlbHMpIGlmIGxhYmVscyBlbHNlIE5vbmUsCiAgICAgICAgImJhbGFuY2VkX2FjY3VyYWN5IjogKHRwciArIHRucikgLyAyIGlmIHRwciBpcyBub3QgTm9uZSBhbmQgdG5yIGlzIG5vdCBOb25lIGVsc2UgTm9uZSwKICAgICAgICAicm9jX2F1YyI6IGF1YyhzY29yZXMsIGxhYmVscyksCiAgICAgICAgInRocmVzaG9sZF9mcm9tX3NkZnZkMl92YWxpZGF0aW9uIjogdGhyZXNob2xkLAogICAgICAgICJjb25mdXNpb25fbWF0cml4IjogeyJ0cnVlX3N5bnRoZXRpYyI6IHRwLCAiZmFsc2Vfc3ludGhldGljIjogZnAsCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgInRydWVfcmVhbCI6IHRuLCAibWlzc2VkX3N5bnRoZXRpYyI6IGZufSwKICAgIH0KCgpkZWYgZmluZF9jbGFzcyhwYXRoOiBQYXRoLCByb290OiBQYXRoKSAtPiBpbnQgfCBOb25lOgogICAgZm9yIHBhcnQgaW4gcmV2ZXJzZWQocGF0aC5yZWxhdGl2ZV90byhyb290KS5wYXJ0c1s6LTFdKToKICAgICAgICBsYWJlbCA9IExBQkVMX0RJUlMuZ2V0KHBhcnQuY2FzZWZvbGQoKSkKICAgICAgICBpZiBsYWJlbCBpcyBub3QgTm9uZToKICAgICAgICAgICAgcmV0dXJuIGxhYmVsCiAgICByZXR1cm4gTm9uZQoKCmRlZiBpbnZlbnRvcnlfYW5kX2R1cGxpY2F0ZV9hdWRpdChkYXRhc2V0X3Jvb3Q6IFBhdGgpIC0+IHR1cGxlW2RpY3QsIGxpc3RbZGljdF1dOgogICAgIiIiQ291bnQgdHJhaW4vdmFsL3Rlc3QgYnkgY2xhc3MgYW5kIGZpbmQgZXhhY3QgZHVwbGljYXRlIGltYWdlIGJ5dGVzIGFjcm9zcyBzcGxpdHMuIiIiCiAgICBjb3VudHM6IGRpY3Rbc3RyLCBDb3VudGVyXSA9IGRlZmF1bHRkaWN0KENvdW50ZXIpCiAgICBoYXNoZXM6IGRpY3Rbc3RyLCBsaXN0W3R1cGxlW3N0ciwgc3RyXV1dID0gZGVmYXVsdGRpY3QobGlzdCkKICAgIHVuY2xhc3NpZmllZCA9IDAKICAgIGZvciBwYXRoIGluIHNvcnRlZChkYXRhc2V0X3Jvb3Qucmdsb2IoIioiKSk6CiAgICAgICAgaWYgbm90IHBhdGguaXNfZmlsZSgpIG9yIHBhdGguc3VmZml4LmNhc2Vmb2xkKCkgbm90IGluIElNQUdFX0VYVEVOU0lPTlM6CiAgICAgICAgICAgIGNvbnRpbnVlCiAgICAgICAgcmVsID0gcGF0aC5yZWxhdGl2ZV90byhkYXRhc2V0X3Jvb3QpCiAgICAgICAgcGFydHMgPSBbcC5jYXNlZm9sZCgpIGZvciBwIGluIHJlbC5wYXJ0c10KICAgICAgICBzcGxpdCA9IG5leHQoKHAgZm9yIHAgaW4gcGFydHMgaWYgcCBpbiB7InRyYWluIiwgInRyYWluaW5nIiwgInZhbCIsICJ2YWxpZCIsICJ2YWxpZGF0aW9uIiwgInRlc3QiLCAidGVzdGluZyJ9KSwgTm9uZSkKICAgICAgICBsYWJlbCA9IGZpbmRfY2xhc3MocGF0aCwgZGF0YXNldF9yb290KQogICAgICAgIGlmIHNwbGl0IGlzIE5vbmUgb3IgbGFiZWwgaXMgTm9uZToKICAgICAgICAgICAgdW5jbGFzc2lmaWVkICs9IDEKICAgICAgICAgICAgY29udGludWUKICAgICAgICBzcGxpdCA9IHsidHJhaW5pbmciOiAidHJhaW4iLCAidmFsaWQiOiAidmFsIiwgInZhbGlkYXRpb24iOiAidmFsIiwgInRlc3RpbmciOiAidGVzdCJ9LmdldChzcGxpdCwgc3BsaXQpCiAgICAgICAgY2xhc3NfbmFtZSA9ICJzeW50aGV0aWMiIGlmIGxhYmVsIGVsc2UgInJlYWwiCiAgICAgICAgY291bnRzW3NwbGl0XVtjbGFzc19uYW1lXSArPSAxCiAgICAgICAgZGlnZXN0ID0gaGFzaGxpYi5zaGEyNTYoKQogICAgICAgIHdpdGggcGF0aC5vcGVuKCJyYiIpIGFzIHN0cmVhbToKICAgICAgICAgICAgZm9yIGNodW5rIGluIGl0ZXIobGFtYmRhOiBzdHJlYW0ucmVhZCgxMDI0ICogMTAyNCksIGIiIik6CiAgICAgICAgICAgICAgICBkaWdlc3QudXBkYXRlKGNodW5rKQogICAgICAgIGhhc2hlc1tkaWdlc3QuaGV4ZGlnZXN0KCldLmFwcGVuZCgoc3BsaXQsIHJlbC5hc19wb3NpeCgpKSkKICAgIGR1cGxpY2F0ZV9ncm91cHMgPSBbXQogICAgZm9yIGRpZ2VzdCwgcm93cyBpbiBoYXNoZXMuaXRlbXMoKToKICAgICAgICBzcGxpdHMgPSBzb3J0ZWQoe3NwbGl0IGZvciBzcGxpdCwgXyBpbiByb3dzfSkKICAgICAgICBpZiBsZW4oc3BsaXRzKSA+IDE6CiAgICAgICAgICAgIGR1cGxpY2F0ZV9ncm91cHMuYXBwZW5kKHsic2hhMjU2IjogZGlnZXN0LCAic3BsaXRzIjogc3BsaXRzLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgImZpbGVzIjogW3BhdGggZm9yIF8sIHBhdGggaW4gcm93c119KQogICAgcmVwb3J0ID0gewogICAgICAgICJjb3VudHNfYnlfc3BsaXQiOiB7c3BsaXQ6IGRpY3Qoc29ydGVkKGNvdW50c1tzcGxpdF0uaXRlbXMoKSkpIGZvciBzcGxpdCBpbiBzb3J0ZWQoY291bnRzKX0sCiAgICAgICAgInVuY2xhc3NpZmllZF9pbWFnZV9maWxlcyI6IHVuY2xhc3NpZmllZCwKICAgICAgICAiZXhhY3RfYnl0ZV9kdXBsaWNhdGVfZ3JvdXBzX2Nyb3NzaW5nX3NwbGl0cyI6IGxlbihkdXBsaWNhdGVfZ3JvdXBzKSwKICAgICAgICAiY3Jvc3Nfc3BsaXRfZHVwbGljYXRlX2V4YW1wbGVzIjogZHVwbGljYXRlX2dyb3Vwc1s6MTAwXSwKICAgICAgICAiZHVwbGljYXRlX3NjYW5fc2NvcGUiOiAiZXhhY3QgU0hBLTI1NiBieXRlIG1hdGNoZXMgb25seTsgZG9lcyBub3QgZGV0ZWN0IHJlLWVuY29kZWQgZnJhbWVzIG9yIHNoYXJlZCBzb3VyY2UgaWRlbnRpdGllcyIsCiAgICB9CiAgICByZXR1cm4gcmVwb3J0LCBkdXBsaWNhdGVfZ3JvdXBzCgoKZGVmIGRpc2NvdmVyX3Rlc3RfcmVjb3Jkcyh0ZXN0X3Jvb3Q6IFBhdGgpIC0+IGxpc3RbZGljdF06CiAgICByZWNvcmRzID0gW10KICAgIGZvciBwYXRoIGluIHNvcnRlZCh0ZXN0X3Jvb3Qucmdsb2IoIioiKSk6CiAgICAgICAgaWYgbm90IHBhdGguaXNfZmlsZSgpIG9yIHBhdGguc3VmZml4LmNhc2Vmb2xkKCkgbm90IGluIElNQUdFX0VYVEVOU0lPTlM6CiAgICAgICAgICAgIGNvbnRpbnVlCiAgICAgICAgbGFiZWwgPSBmaW5kX2NsYXNzKHBhdGgsIHRlc3Rfcm9vdCkKICAgICAgICBpZiBsYWJlbCBpcyBOb25lOgogICAgICAgICAgICBjb250aW51ZQogICAgICAgIHJlY29yZHMuYXBwZW5kKHsicGF0aCI6IHBhdGgsICJsYWJlbCI6IGxhYmVsLCAicmVsYXRpdmVfcGF0aCI6IHBhdGgucmVsYXRpdmVfdG8odGVzdF9yb290KS5hc19wb3NpeCgpfSkKICAgIHJldHVybiByZWNvcmRzCgoKY2xhc3MgSW1hZ2VEYXRhc2V0KERhdGFzZXQpOgogICAgZGVmIF9faW5pdF9fKHNlbGYsIHJlY29yZHM6IGxpc3RbZGljdF0sIHNpemU6IGludCk6CiAgICAgICAgc2VsZi5yZWNvcmRzID0gcmVjb3JkcwogICAgICAgIHNlbGYudHJhbnNmb3JtID0gdHJhbnNmb3Jtcy5Db21wb3NlKFsKICAgICAgICAgICAgdHJhbnNmb3Jtcy5SZXNpemUoKHNpemUsIHNpemUpKSwKICAgICAgICAgICAgdHJhbnNmb3Jtcy5Ub1RlbnNvcigpLAogICAgICAgICAgICB0cmFuc2Zvcm1zLk5vcm1hbGl6ZSgoLjQ4NSwgLjQ1NiwgLjQwNiksICguMjI5LCAuMjI0LCAuMjI1KSksCiAgICAgICAgXSkKCiAgICBkZWYgX19sZW5fXyhzZWxmKToKICAgICAgICByZXR1cm4gbGVuKHNlbGYucmVjb3JkcykKCiAgICBkZWYgX19nZXRpdGVtX18oc2VsZiwgaWR4KToKICAgICAgICB3aXRoIEltYWdlLm9wZW4oc2VsZi5yZWNvcmRzW2lkeF1bInBhdGgiXSkgYXMgaW1hZ2U6CiAgICAgICAgICAgIHRlbnNvciA9IHNlbGYudHJhbnNmb3JtKGltYWdlLmNvbnZlcnQoIlJHQiIpKQogICAgICAgIHJldHVybiB0ZW5zb3IsIGlkeAoKCmRlZiBkaXNjb3Zlcl9mb2xkcyhjdl9yb290OiBQYXRoKSAtPiBsaXN0W3R1cGxlW1BhdGgsIFBhdGgsIFBhdGhdXToKICAgIGZvbGRzID0gW10KICAgIGZvciBmb2xkX2RpciBpbiBzb3J0ZWQoY3Zfcm9vdC5nbG9iKCJmb2xkXyoiKSk6CiAgICAgICAgY2hlY2twb2ludCA9IGZvbGRfZGlyIC8gIm11bHRpbW9kYWwiIC8gImJlc3RfYXZfbW9kZWwucHQiCiAgICAgICAgdGhyZXNob2xkX3BhdGggPSBmb2xkX2RpciAvICJ0aHJlc2hvbGRfdmlzdWFsLmpzb24iCiAgICAgICAgaWYgY2hlY2twb2ludC5pc19maWxlKCkgYW5kIHRocmVzaG9sZF9wYXRoLmlzX2ZpbGUoKToKICAgICAgICAgICAgZm9sZHMuYXBwZW5kKChmb2xkX2RpciwgY2hlY2twb2ludCwgdGhyZXNob2xkX3BhdGgpKQogICAgaWYgbGVuKGZvbGRzKSA8IDM6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcihmIkV4cGVjdGVkIGF0IGxlYXN0IHRocmVlIGZvbGQgY2hlY2twb2ludHMgYW5kIHZpc3VhbCB0aHJlc2hvbGRzIHVuZGVyIHtjdl9yb290fSIpCiAgICByZXR1cm4gZm9sZHMKCgpkZWYgcHJlZGljdChtb2RlbCwgbG9hZGVyLCBkZXZpY2U6IHRvcmNoLmRldmljZSwgY291bnQ6IGludCkgLT4gbGlzdFtmbG9hdF06CiAgICByZXN1bHQgPSBbbWF0aC5uYW5dICogY291bnQKICAgIHdpdGggdG9yY2guaW5mZXJlbmNlX21vZGUoKToKICAgICAgICBmb3IgaW1hZ2VzLCBpbmRpY2VzIGluIGxvYWRlcjoKICAgICAgICAgICAgbG9naXRzID0gbW9kZWwudmlzdWFsX2hlYWQobW9kZWwudmlzdWFsKGltYWdlcy50byhkZXZpY2UpKSkuc3F1ZWV6ZSgxKQogICAgICAgICAgICBmb3IgaWR4LCB2YWx1ZSBpbiB6aXAoaW5kaWNlcy50b2xpc3QoKSwgdG9yY2guc2lnbW9pZChsb2dpdHMpLmNwdSgpLnRvbGlzdCgpKToKICAgICAgICAgICAgICAgIHJlc3VsdFtpZHhdID0gZmxvYXQodmFsdWUpCiAgICBpZiBhbnkobm90IG1hdGguaXNmaW5pdGUodmFsdWUpIGZvciB2YWx1ZSBpbiByZXN1bHQpOgogICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcigiTm90IGFsbCB0ZXN0IGltYWdlcyByZWNlaXZlZCBhIGZpbml0ZSBzY29yZSIpCiAgICByZXR1cm4gcmVzdWx0CgoKZGVmIG1haW4oKSAtPiBpbnQ6CiAgICBwYXJzZXIgPSBhcmdwYXJzZS5Bcmd1bWVudFBhcnNlcihkZXNjcmlwdGlvbj1fX2RvY19fKQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS1kYXRhc2V0LXJvb3QiLCB0eXBlPVBhdGgsIHJlcXVpcmVkPVRydWUsIGhlbHA9IkV4dHJhY3RlZCBLYWdnbGUgZGF0YXNldCByb290IGNvbnRhaW5pbmcgdHJhaW4vdmFsL3Rlc3QiKQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS10ZXN0LXJvb3QiLCB0eXBlPVBhdGgsIHJlcXVpcmVkPVRydWUsIGhlbHA9IlRoZSB0ZXN0IHNwbGl0IGRpcmVjdG9yeSBvbmx5OyBlLmcuIGRhdGFzZXQvdGVzdCIpCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLWN2LXJvb3QiLCB0eXBlPVBhdGgsIHJlcXVpcmVkPVRydWUsIGhlbHA9IlNERlZEMiBncm91cF9jdiBkaXJlY3Rvcnkgd2l0aCBmaXZlIGZvbGQgY2hlY2twb2ludHMiKQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS1vdXRwdXQtZGlyIiwgdHlwZT1QYXRoLCByZXF1aXJlZD1UcnVlKQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS1iYXRjaC1zaXplIiwgdHlwZT1pbnQsIGRlZmF1bHQ9NjQpCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLXdvcmtlcnMiLCB0eXBlPWludCwgZGVmYXVsdD0yKQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS1kZXZpY2UiLCBjaG9pY2VzPSgiYXV0byIsICJjcHUiLCAiY3VkYSIsICJtcHMiKSwgZGVmYXVsdD0iYXV0byIpCiAgICBhcmdzID0gcGFyc2VyLnBhcnNlX2FyZ3MoKQogICAgdHJ5OgogICAgICAgIGRhdGFzZXRfcm9vdCwgdGVzdF9yb290LCBjdl9yb290LCBvdXRwdXQgPSAocC5leHBhbmR1c2VyKCkucmVzb2x2ZSgpIGZvciBwIGluCiAgICAgICAgICAgIChhcmdzLmRhdGFzZXRfcm9vdCwgYXJncy50ZXN0X3Jvb3QsIGFyZ3MuY3Zfcm9vdCwgYXJncy5vdXRwdXRfZGlyKSkKICAgICAgICBpZiBub3QgZGF0YXNldF9yb290LmlzX2RpcigpIG9yIG5vdCB0ZXN0X3Jvb3QuaXNfZGlyKCkgb3Igbm90IGN2X3Jvb3QuaXNfZGlyKCk6CiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoImRhdGFzZXQtcm9vdCwgdGVzdC1yb290LCBhbmQgY3Ytcm9vdCBtdXN0IGJlIGV4aXN0aW5nIGRpcmVjdG9yaWVzIikKICAgICAgICBpZiBhcmdzLmJhdGNoX3NpemUgPCAxIG9yIGFyZ3Mud29ya2VycyA8IDA6CiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoImJhdGNoLXNpemUgbXVzdCBiZSBwb3NpdGl2ZSBhbmQgd29ya2VycyBub25uZWdhdGl2ZSIpCiAgICAgICAgaWYgb3V0cHV0LmV4aXN0cygpIGFuZCAobm90IG91dHB1dC5pc19kaXIoKSBvciBhbnkob3V0cHV0Lml0ZXJkaXIoKSkpOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKGYib3V0cHV0IGRpcmVjdG9yeSBtdXN0IGJlIG5ldyBvciBlbXB0eToge291dHB1dH0iKQogICAgICAgIHRyeToKICAgICAgICAgICAgdGVzdF9yb290LnJlbGF0aXZlX3RvKGRhdGFzZXRfcm9vdCkKICAgICAgICBleGNlcHQgVmFsdWVFcnJvciBhcyBleGM6CiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoInRlc3Qtcm9vdCBtdXN0IGJlIGluc2lkZSBkYXRhc2V0LXJvb3QiKSBmcm9tIGV4YwogICAgICAgIGlmIG5vdCBhbnkocGFydC5jYXNlZm9sZCgpIGluIHsidGVzdCIsICJ0ZXN0aW5nIn0gZm9yIHBhcnQgaW4gdGVzdF9yb290LnBhcnRzKToKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiUmVmdXNpbmcgZXZhbHVhdGlvbjogdGVzdC1yb290IHBhdGggbXVzdCBleHBsaWNpdGx5IGlkZW50aWZ5IHRoZSB0ZXN0IHNwbGl0IikKICAgICAgICBkZXZpY2VfbmFtZSA9IGFyZ3MuZGV2aWNlCiAgICAgICAgaWYgZGV2aWNlX25hbWUgPT0gImF1dG8iOgogICAgICAgICAgICBkZXZpY2VfbmFtZSA9ICJjdWRhIiBpZiB0b3JjaC5jdWRhLmlzX2F2YWlsYWJsZSgpIGVsc2UgIm1wcyIgaWYgaGFzYXR0cih0b3JjaC5iYWNrZW5kcywgIm1wcyIpIGFuZCB0b3JjaC5iYWNrZW5kcy5tcHMuaXNfYXZhaWxhYmxlKCkgZWxzZSAiY3B1IgogICAgICAgIGlmIGRldmljZV9uYW1lID09ICJjdWRhIiBhbmQgbm90IHRvcmNoLmN1ZGEuaXNfYXZhaWxhYmxlKCk6CiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoIkNVREEgcmVxdWVzdGVkIGJ1dCB1bmF2YWlsYWJsZSIpCiAgICAgICAgaWYgZGV2aWNlX25hbWUgPT0gIm1wcyIgYW5kIG5vdCAoaGFzYXR0cih0b3JjaC5iYWNrZW5kcywgIm1wcyIpIGFuZCB0b3JjaC5iYWNrZW5kcy5tcHMuaXNfYXZhaWxhYmxlKCkpOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJNUFMgcmVxdWVzdGVkIGJ1dCB1bmF2YWlsYWJsZSIpCiAgICAgICAgZGV2aWNlID0gdG9yY2guZGV2aWNlKGRldmljZV9uYW1lKQoKICAgICAgICBpbnZlbnRvcnksIGR1cGxpY2F0ZV9ncm91cHMgPSBpbnZlbnRvcnlfYW5kX2R1cGxpY2F0ZV9hdWRpdChkYXRhc2V0X3Jvb3QpCiAgICAgICAgcmVjb3JkcyA9IGRpc2NvdmVyX3Rlc3RfcmVjb3Jkcyh0ZXN0X3Jvb3QpCiAgICAgICAgbGFiZWxzID0gW3Jvd1sibGFiZWwiXSBmb3Igcm93IGluIHJlY29yZHNdCiAgICAgICAgaWYgbm90IHJlY29yZHMgb3Igbm90IGFueShsYWJlbHMpIG9yIGFsbChsYWJlbHMpOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJUZXN0IHNwbGl0IG5lZWRzIGF0IGxlYXN0IG9uZSBpbWFnZSBpbiBib3RoIHJlYWwgYW5kIGZha2UgY2xhc3MgZm9sZGVycyIpCiAgICAgICAgZm9sZHMgPSBkaXNjb3Zlcl9mb2xkcyhjdl9yb290KQogICAgICAgIG91dHB1dC5ta2RpcihwYXJlbnRzPVRydWUsIGV4aXN0X29rPVRydWUpCiAgICAgICAgZm9sZF9yZXBvcnRzLCBwcmVkaWN0aW9ucyA9IFtdLCBbXQogICAgICAgIGZvciBmb2xkX2RpciwgY2hlY2twb2ludF9wYXRoLCB0aHJlc2hvbGRfcGF0aCBpbiBmb2xkczoKICAgICAgICAgICAgdGhyZXNob2xkX2RhdGEgPSBqc29uLmxvYWRzKHRocmVzaG9sZF9wYXRoLnJlYWRfdGV4dChlbmNvZGluZz0idXRmLTgiKSkKICAgICAgICAgICAgdGhyZXNob2xkID0gZmxvYXQodGhyZXNob2xkX2RhdGFbInNlbGVjdGVkX3RocmVzaG9sZCJdKQogICAgICAgICAgICBpZiBub3QgbWF0aC5pc2Zpbml0ZSh0aHJlc2hvbGQpIG9yIG5vdCAwIDw9IHRocmVzaG9sZCA8PSAxOgogICAgICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcihmIkludmFsaWQgdGhyZXNob2xkIGluIHt0aHJlc2hvbGRfcGF0aH0iKQogICAgICAgICAgICBjaGVja3BvaW50ID0gdG9yY2gubG9hZChjaGVja3BvaW50X3BhdGgsIG1hcF9sb2NhdGlvbj0iY3B1Iiwgd2VpZ2h0c19vbmx5PUZhbHNlKQogICAgICAgICAgICBpbWFnZV9zaXplID0gaW50KGNoZWNrcG9pbnQuZ2V0KCJpbWFnZV9zaXplIiwgMjI0KSkKICAgICAgICAgICAgbW9kZWwgPSBidWlsZF9tb2RlbChwcmV0cmFpbmVkX3Zpc3VhbD1GYWxzZSkudG8oZGV2aWNlKQogICAgICAgICAgICBtb2RlbC5sb2FkX3N0YXRlX2RpY3QoY2hlY2twb2ludFsic3RhdGVfZGljdCJdKQogICAgICAgICAgICBtb2RlbC5ldmFsKCkKICAgICAgICAgICAgbG9hZGVyID0gRGF0YUxvYWRlcihJbWFnZURhdGFzZXQocmVjb3JkcywgaW1hZ2Vfc2l6ZSksIGJhdGNoX3NpemU9YXJncy5iYXRjaF9zaXplLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIHNodWZmbGU9RmFsc2UsIG51bV93b3JrZXJzPWFyZ3Mud29ya2VycywgcGluX21lbW9yeT1kZXZpY2UudHlwZSA9PSAiY3VkYSIpCiAgICAgICAgICAgIHNjb3JlcyA9IHByZWRpY3QobW9kZWwsIGxvYWRlciwgZGV2aWNlLCBsZW4ocmVjb3JkcykpCiAgICAgICAgICAgIHJlcG9ydCA9IHsiZm9sZCI6IGZvbGRfZGlyLm5hbWUsICoqc2NvcmVfbWV0cmljcyhzY29yZXMsIGxhYmVscywgdGhyZXNob2xkKX0KICAgICAgICAgICAgZm9sZF9yZXBvcnRzLmFwcGVuZChyZXBvcnQpCiAgICAgICAgICAgIGZvciByb3csIHNjb3JlIGluIHppcChyZWNvcmRzLCBzY29yZXMpOgogICAgICAgICAgICAgICAgcHJlZGljdGlvbnMuYXBwZW5kKHsiZm9sZCI6IGZvbGRfZGlyLm5hbWUsICJ0ZXN0X2ltYWdlIjogcm93WyJyZWxhdGl2ZV9wYXRoIl0sCiAgICAgICAgICAgICAgICAgICAgImxhYmVsIjogInN5bnRoZXRpYyIgaWYgcm93WyJsYWJlbCJdIGVsc2UgInJlYWwiLCAic2NvcmVfc3ludGhldGljIjogc2NvcmUsCiAgICAgICAgICAgICAgICAgICAgInRocmVzaG9sZF9mcm9tX3NkZnZkMl92YWxpZGF0aW9uIjogdGhyZXNob2xkLAogICAgICAgICAgICAgICAgICAgICJwcmVkaWN0aW9uIjogInN5bnRoZXRpYyIgaWYgc2NvcmUgPj0gdGhyZXNob2xkIGVsc2UgInJlYWwifSkKICAgICAgICAgICAgcHJpbnQoZiJTY29yZWQge2ZvbGRfZGlyLm5hbWV9OiB7bGVuKHJlY29yZHMpfSB0ZXN0IGltYWdlczsgUk9DIEFVQz17cmVwb3J0Wydyb2NfYXVjJ119IiwgZmx1c2g9VHJ1ZSkKICAgICAgICAgICAgZGVsIG1vZGVsCiAgICAgICAgICAgIGlmIGRldmljZS50eXBlID09ICJjdWRhIjoKICAgICAgICAgICAgICAgIHRvcmNoLmN1ZGEuZW1wdHlfY2FjaGUoKQoKICAgICAgICBzdW1tYXJ5ID0ge30KICAgICAgICBmb3IgbWV0cmljIGluICgiYWNjdXJhY3kiLCAiYmFsYW5jZWRfYWNjdXJhY3kiLCAicm9jX2F1YyIpOgogICAgICAgICAgICB2YWx1ZXMgPSBbcm93W21ldHJpY10gZm9yIHJvdyBpbiBmb2xkX3JlcG9ydHMgaWYgcm93W21ldHJpY10gaXMgbm90IE5vbmVdCiAgICAgICAgICAgIHN1bW1hcnlbbWV0cmljXSA9IHsibWVhbiI6IHN0YXRpc3RpY3MubWVhbih2YWx1ZXMpIGlmIHZhbHVlcyBlbHNlIE5vbmUsCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAic3RkZXYiOiBzdGF0aXN0aWNzLnN0ZGV2KHZhbHVlcykgaWYgbGVuKHZhbHVlcykgPiAxIGVsc2UgMC4wIGlmIHZhbHVlcyBlbHNlIE5vbmUsCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAiZm9sZF92YWx1ZXMiOiB2YWx1ZXN9CiAgICAgICAgcmVwb3J0ID0gewogICAgICAgICAgICAic2NoZW1hX3ZlcnNpb24iOiAiMS4wIiwKICAgICAgICAgICAgImRhdGFzZXQiOiAiS2FnZ2xlIGRlZXBmYWtlLWRhdGFzZXQgKEZhY2VGb3JlbnNpY3MrKyBDMjMgYW5kIENlbGViLURGIHYyIHBlciBwdWJsaXNoZXIgZGVzY3JpcHRpb24pIiwKICAgICAgICAgICAgImV2YWx1YXRpb25fc3BsaXQiOiAidGVzdCBvbmx5IiwKICAgICAgICAgICAgInRyYWluaW5nX3BlcmZvcm1lZCI6IEZhbHNlLAogICAgICAgICAgICAiZXh0ZXJuYWxfZXZhbHVhdGlvbl9zdGF0dXMiOiAiZXhwbG9yYXRvcnlfdGVzdF9zcGxpdF93aXRoX3VudmVyaWZpZWRfdXBzdHJlYW1fbGluZWFnZSIsCiAgICAgICAgICAgICJydW5fZGV2aWNlIjogc3RyKGRldmljZSksCiAgICAgICAgICAgICJzY29yZV91bml0IjogImluZGl2aWR1YWwgZmFjZS1mcmFtZSBpbWFnZTsgZnJhbWVzIG1heSBiZSBjb3JyZWxhdGVkIGFuZCBhcmUgbm90IGluZGVwZW5kZW50IHZpZGVvcyIsCiAgICAgICAgICAgICJpbnZlbnRvcnlfYW5kX2V4YWN0X2R1cGxpY2F0ZV9hdWRpdCI6IGludmVudG9yeSwKICAgICAgICAgICAgImZvbGRzIjogZm9sZF9yZXBvcnRzLAogICAgICAgICAgICAiZm9sZF9tZWFuIjogc3VtbWFyeSwKICAgICAgICAgICAgImxpbWl0YXRpb25zIjogWwogICAgICAgICAgICAgICAgIlRoZSBkYXRhc2V0IHB1Ymxpc2hlcidzIHNwbGl0IGFuZCBzb3VyY2UtdmlkZW8gZGlzam9pbnRuZXNzIGNsYWltcyB3ZXJlIG5vdCBpbmRlcGVuZGVudGx5IHZlcmlmaWVkLiIsCiAgICAgICAgICAgICAgICAiVGhlIGV4YWN0IGR1cGxpY2F0ZSBzY2FuIGRldGVjdHMgaWRlbnRpY2FsIGZpbGUgYnl0ZXMgYWNyb3NzIHNwbGl0cywgbm90IHJlLWVuY29kZWQgY29waWVzLCBuZWFyLWR1cGxpY2F0ZXMsIHNoYXJlZCBpZGVudGl0aWVzLCBvciByZWxhdGVkIGZyYW1lcy4iLAogICAgICAgICAgICAgICAgIlRoZSBLYWdnbGUgYXJjaGl2ZSBjb250YWlucyBmYWNlLWZyYW1lIGltYWdlczsgdGhpcyB2aXN1YWwtb25seSBldmFsdWF0aW9uIGRvZXMgbm90IGFzc2VzcyBhdWRpbyBvciB0ZW1wb3JhbCB2aWRlbyBldmlkZW5jZS4iLAogICAgICAgICAgICAgICAgIk1ldHJpY3MgYXJlIGltYWdlLXdlaWdodGVkOyBtdWx0aXBsZSBjb3JyZWxhdGVkIGZyYW1lcyBtYXkgY29tZSBmcm9tIHRoZSBzYW1lIHNvdXJjZSB2aWRlby4iLAogICAgICAgICAgICAgICAgIk5vIG1vZGVsIHdhcyB0cmFpbmVkIG9yIGNhbGlicmF0ZWQgb24gdGhpcyBkYXRhc2V0LiBGb2xkLXNwZWNpZmljIHRocmVzaG9sZHMgY29tZSBvbmx5IGZyb20gU0RGVkQyIHZhbGlkYXRpb24gZGF0YS4iLAogICAgICAgICAgICBdLAogICAgICAgIH0KICAgICAgICAob3V0cHV0IC8gImV4dGVybmFsX2V2YWx1YXRpb24uanNvbiIpLndyaXRlX3RleHQoanNvbi5kdW1wcyhyZXBvcnQsIGluZGVudD0yKSArICJcbiIsIGVuY29kaW5nPSJ1dGYtOCIpCiAgICAgICAgd2l0aCAob3V0cHV0IC8gImV4dGVybmFsX3ByZWRpY3Rpb25zLmNzdiIpLm9wZW4oInciLCBuZXdsaW5lPSIiLCBlbmNvZGluZz0idXRmLTgiKSBhcyBzdHJlYW06CiAgICAgICAgICAgIHdyaXRlciA9IGNzdi5EaWN0V3JpdGVyKHN0cmVhbSwgZmllbGRuYW1lcz1saXN0KHByZWRpY3Rpb25zWzBdKSkKICAgICAgICAgICAgd3JpdGVyLndyaXRlaGVhZGVyKCkKICAgICAgICAgICAgd3JpdGVyLndyaXRlcm93cyhwcmVkaWN0aW9ucykKICAgICAgICBwcmludChqc29uLmR1bXBzKHsic3RhdHVzIjogcmVwb3J0WyJleHRlcm5hbF9ldmFsdWF0aW9uX3N0YXR1cyJdLCAiaW52ZW50b3J5IjogaW52ZW50b3J5LAogICAgICAgICAgICAgICAgICAgICAgICAgICJmb2xkX21lYW4iOiBzdW1tYXJ5LCAidGVzdF9pbWFnZXMiOiBsZW4ocmVjb3Jkcyl9LCBpbmRlbnQ9MikpCiAgICAgICAgcHJpbnQoZiJFeHRlcm5hbCBldmFsdWF0aW9uIGFydGlmYWN0czoge291dHB1dH0iKQogICAgICAgIHJldHVybiAwCiAgICBleGNlcHQgKE9TRXJyb3IsIFZhbHVlRXJyb3IsIEtleUVycm9yLCBqc29uLkpTT05EZWNvZGVFcnJvciwgUnVudGltZUVycm9yKSBhcyBleGM6CiAgICAgICAgcHJpbnQoZiJFcnJvcjoge2V4Y30iLCBmaWxlPXN5cy5zdGRlcnIpCiAgICAgICAgcmV0dXJuIDIKCgppZiBfX25hbWVfXyA9PSAiX19tYWluX18iOgogICAgcmFpc2UgU3lzdGVtRXhpdChtYWluKCkpCg=='
SOURCE_ZIP = WORK / 'poor_guys_deepfake_colab_source.zip'
SOURCE_ZIP.write_bytes(base64.b64decode(SOURCE_B64.read_bytes().strip()))
PROJECT_ROOT = WORK / 'project_source'
PROJECT_ROOT.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(SOURCE_ZIP) as zf:
    zf.extractall(PROJECT_ROOT)
PROJECT = PROJECT_ROOT / 'poor_guys_deepfake_project'
sys.path.insert(0, str(PROJECT / 'multimodal'))
EVALUATOR_PATH = PROJECT / 'multimodal' / 'evaluate_kaggle_external.py'
EVALUATOR_PATH.write_bytes(base64.b64decode(EVALUATOR_B64))
print('Evaluator and model source are ready:', EVALUATOR_PATH)


In [ ]:
# Evaluate each existing fold checkpoint once on the untouched test split.
OUTPUT = DEST / f'external_eval_{time.strftime("%Y%m%dT%H%M%SZ", time.gmtime())}'
cmd = [sys.executable, str(EVALUATOR_PATH),
       '--dataset-root', str(DATASET_ROOT), '--test-root', str(TEST_ROOT),
       '--cv-root', str(CV_ROOT), '--output-dir', str(OUTPUT), '--device', 'auto']
print('Starting test-only external evaluation...')
subprocess.run(cmd, cwd=PROJECT, check=True)
print('Report:', OUTPUT / 'external_evaluation.json')
print('Predictions:', OUTPUT / 'external_predictions.csv')


In [ ]:
# Recalculate metrics after counting byte-identical test images once.
# This reuses saved predictions, without running inference again.
import csv, hashlib, json, statistics
from collections import defaultdict
from evaluate_kaggle_external import score_metrics

prediction_path = OUTPUT / 'external_predictions.csv'
image_hashes = {}
for image_path in sorted(TEST_ROOT.rglob('*')):
    if image_path.is_file() and image_path.suffix.lower() in {'.jpg', '.jpeg', '.png', '.webp', '.bmp'}:
        h = hashlib.sha256()
        with image_path.open('rb') as stream:
            for chunk in iter(lambda: stream.read(1024 * 1024), b''):
                h.update(chunk)
        image_hashes[image_path.relative_to(TEST_ROOT).as_posix()] = h.hexdigest()

unique = defaultdict(dict)
original_count = defaultdict(int)
with prediction_path.open(newline='', encoding='utf-8') as stream:
    for row in csv.DictReader(stream):
        digest = image_hashes.get(row['test_image'])
        if digest is None:
            raise RuntimeError(f"Prediction has no test image for hashing: {row['test_image']}")
        entry = {'label': 1 if row['label'] == 'synthetic' else 0,
                 'score': float(row['score_synthetic']),
                 'threshold': float(row['threshold_from_sdfvd2_validation']),
                 'path': row['test_image']}
        prior = unique[row['fold']].get(digest)
        if prior and prior['label'] != entry['label']:
            raise RuntimeError(f"Exact-byte duplicate has conflicting labels: {prior['path']} and {entry['path']}")
        unique[row['fold']].setdefault(digest, entry)
        original_count[row['fold']] += 1

deduplicated_folds = []
for fold, rows_by_hash in sorted(unique.items()):
    rows = list(rows_by_hash.values())
    labels = [row['label'] for row in rows]
    scores = [row['score'] for row in rows]
    thresholds = {row['threshold'] for row in rows}
    if len(thresholds) != 1:
        raise RuntimeError(f"Fold {fold} contains multiple thresholds")
    deduplicated_folds.append({'fold': fold, 'duplicate_test_rows_removed': original_count[fold] - len(rows),
                               **score_metrics(scores, labels, thresholds.pop())})

fold_mean = {}
for metric in ('accuracy', 'balanced_accuracy', 'roc_auc'):
    values = [row[metric] for row in deduplicated_folds if row[metric] is not None]
    fold_mean[metric] = {'mean': statistics.mean(values) if values else None,
                         'stdev': statistics.stdev(values) if len(values) > 1 else 0.0 if values else None,
                         'fold_values': values}
deduplicated_report = {
    'evaluation': 'test split; one image per exact SHA-256 content per fold',
    'prediction_rows_before_deduplication': sum(original_count.values()),
    'prediction_rows_after_deduplication': sum(len(rows) for rows in unique.values()),
    'folds': deduplicated_folds,
    'fold_mean': fold_mean,
    'interpretation': 'Exact-byte deduplication removes repeated image weighting only; source-video and identity independence remain unverified.',
}
dedup_path = OUTPUT / 'exact_duplicate_deduplicated_evaluation.json'
dedup_path.write_text(json.dumps(deduplicated_report, indent=2) + '\n', encoding='utf-8')
print(json.dumps({'deduplicated_test_counts': [row['count'] for row in deduplicated_folds], 'fold_mean': fold_mean}, indent=2))
print('Deduplicated report:', dedup_path)


## How to read the report

The fold mean and spread summarize five separately trained SDFVD2 checkpoints evaluated with their own SDFVD2 validation thresholds. The Kaggle test split was not used for training or threshold selection. The report includes exact-file duplicate checks across train/validation/test; this does not prove source-video or identity separation. Metrics are frame-image weighted, not video weighted, and should be described as exploratory cross-dataset visual results until source lineage and upstream licenses are independently confirmed. The final code cell recalculates metrics after removing repeated byte-identical test images, using the saved scores and no new inference.
